# Function Generator

You can use your ALPACA like the function generator on the caddy. The <font color="#a02d9c">Helper Pico</font> can generate signals independently from your code in <font color="#2d91a0">Student Pico</font>.

The <font color="#c81d7a">**function generator**</font> lives on the <font color="#a02d9c">Helper Pico</font> and outputs analog waveforms for you. You drive it from <font color="#2d91a0">Student Pico</font> through the `helper` module, which sends the commands over the link between the two Picos.

```{tip}
Once started, your waveform keeps playing on its own until you stop it, so the rest of your code can do other things in the meantime.
```

```{danger}
**Only one waveform plays at a time.** Starting a new waveform automatically stops the previous one.
```

## Standard Waveforms

`helper.function_generator` gives you four ready-made waveforms. Each one plays inside a `with` block: it **starts** when you enter the block and **stops** (output goes back to 0) when you leave.

- `function_generator.sine(frequency_hz, amplitude_vpp)`
- `function_generator.block(frequency_hz, amplitude_vpp, duty_cycle=0.5)` (square wave)
- `function_generator.triangle(frequency_hz, amplitude_vpp)`
- `function_generator.sawtooth(frequency_hz, amplitude_vpp)` (rising ramp)

```py
@pico.task
def generate_wave():
    import helper

    # Plays a 2 kHz, 2.2 Vpp sine while inside the block, then stops
    with helper.function_generator.sine(2_000, 2.2):
        ...

    # Square wave 1 kHz, 3.0 Vpp with a 25% duty cycle
    with helper.function_generator.block(1_000, 3.0, duty_cycle=0.25):
        ...
```

### Parameters

| Parameter | Meaning |
|-----------|---------|
| <font color="#0057c0">**`frequency_hz`**</font> | The frequency in **whole Hz**, must be `>= 1`. |
| <font color="#c05a00">**`amplitude_vpp`**</font> | The amplitude in **volts peak-to-peak**. Full scale is 3.3V, so anything above **3.3 Vpp clips**. |
| <font color="#008060">**`duty_cycle`**</font> | Only for `block`: the fraction of the period that is HIGH, must be `0 < duty_cycle < 1`. The other waveforms ignore it. |

### Controlling playback manually

Instead of a `with` block you can start and stop the waveform yourself. `.start()` returns the **actual frequency achieved** in Hz, which can differ slightly from what you asked for.

```py
@pico.task
def manual_control():
    import helper

    wave = helper.function_generator.triangle(500, 1.0)

    actual = wave.start()   # begin playback, returns the real frequency
    print(actual)           # e.g. 499.7

    # ... waveform keeps looping on its own here ...

    wave.stop()             # stop, output goes to 0
```

```{tip}
Lower frequencies result in **sharper signals**.
```

:::{admonition} For the curious: Arbitrary Waveforms
:class: deep-dive, dropdown

When none of the four shapes fit, you can build your **own waveform** from a list of values. This is the same idea as [supplying a custom waveform to the DAC](./signals.ipynb#ac-output): there you pass a *function* of `t` describing one period, and here you pass the **values themselves** for one period.

```py
actual_rate = helper.awg_r2r(samples, sample_rate_hz)
```

| Parameter | Meaning |
|-----------|---------|
| <font color="#c81d7a">**`samples`**</font> | A list of numbers describing **one period** of your signal. Each number sets the output level, from `0` (0V) up to `1023` (~3.3V). Up to 4096 values, played on a loop. |
| <font color="#0057c0">**`sample_rate_hz`**</font> | How many values per second to play. The call returns the rate it actually managed, which is lower if you ask for something too fast. |

Because the whole list is one period, the frequency you get is:

$$
\textcolor{#0057c0}{f_{\text{out}}} = \frac{\textcolor{#0057c0}{\text{sample_rate_hz}}}{\text{len(samples)}}
$$

So **you** choose both the shape of the list and the rate that gives the frequency you want. To turn a voltage into a level, use `int(value / 3.3 * 1023)`.

```py
@pico.task
def custom_sine():
    import helper
    import math

    N = 256                       # values in one period
    freq_hz = 1_000               # desired output frequency

    # One period of a sine, as levels between 0 and 1023
    samples = [
        int((math.sin(2 * math.pi * i / N) + 1) / 2 * 1023)
        for i in range(N)
    ]

    # sample_rate = freq * N  ->  f_out = sample_rate / N = freq
    helper.awg_r2r(samples, freq_hz * N)

    # ... the waveform keeps playing until you stop it ...

    helper.awg_r2r_stop()         # stop, output goes to 0
```

### Adding an offset

Standard waves are always centered halfway up the range, but with a custom waveform you can shift the whole signal up or down:

```py
# magnitude is 0.0 to 1.0, negative picks the direction
helper.awg_r2r_offset(0.25)                 # shift up
helper.awg_r2r_offset(0.25, negative=True)  # shift down
```

### Good to know

- **One waveform at a time:** starting a new one stops the current one.
- **At most 4096 values** in your list.
- Values above ~3.3V clip at the top of the range.
- Very high sample rates get reduced, so read back the rate the call returns when it matters.

:::